In [1]:
# Cell 1: Import libraries and verify setup
import pandas as pd
import numpy as np
import sqlite3
import matplotlib.pyplot as plt
import seaborn as sns
import os

print("All libraries imported successfully!")
print(f"Pandas version: {pd.__version__}")


All libraries imported successfully!
Pandas version: 3.0.3


In [2]:
# Cell 2: Load the dataset
file_path = '../data/WA_Fn-UseC_-Telco-Customer-Churn.csv'

# Check if file exists
if os.path.exists(file_path):
    df = pd.read_csv(file_path)
    print(f"Dataset loaded successfully!")
    print(f"Shape: {df.shape[0]} rows, {df.shape[1]} columns")
    print(f"Column names: {list(df.columns)[:5]}...")  # First 5 columns only
else:
    print(f"File not found at: {file_path}")
    print("Let me check what's in the data folder:")
    print(os.listdir('../data'))

Dataset loaded successfully!
Shape: 7043 rows, 21 columns
Column names: ['customerID', 'gender', 'SeniorCitizen', 'Partner', 'Dependents']...


In [4]:
# Cell 3: First look at the data
print("First 5 rows of the dataset:")
df.head()

First 5 rows of the dataset:


,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,...,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,7590-VHVEG,Female,0,Yes,No,1,No,No phone service,DSL,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No
1,5575-GNVDE,Male,0,No,No,34,Yes,No,DSL,Yes,...,Yes,No,No,No,One year,No,Mailed check,56.95,1889.5,No
2,3668-QPYBK,Male,0,No,No,2,Yes,No,DSL,Yes,...,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes
3,7795-CFOCW,Male,0,No,No,45,No,No phone service,DSL,Yes,...,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,No
4,9237-HQITU,Female,0,No,No,2,Yes,No,Fiber optic,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,Yes


In [3]:
# Cell 4: Create SQLite database
import sqlite3

# Connect to database (creates file automatically)
conn = sqlite3.connect('../data/churn.db')

# Write the dataframe to SQL table
df.to_sql('customers', conn, if_exists='replace', index=False)
print("Database 'churn.db' created with table 'customers'")

# Verify it worked
query = "SELECT name FROM sqlite_master WHERE type='table';"
tables = pd.read_sql(query, conn)
print(f"Tables in database: {tables['name'].tolist()}")

# Close connection
conn.close()

Database 'churn.db' created with table 'customers'
Tables in database: ['customers']


In [4]:
# Cell 5: SQL Query - Churn rate by contract type
conn = sqlite3.connect('../data/churn.db')

query = """
SELECT 
    Contract,
    COUNT(*) as total_customers,
    SUM(CASE WHEN Churn = 'Yes' THEN 1 ELSE 0 END) as churned_customers,
    ROUND(100.0 * SUM(CASE WHEN Churn = 'Yes' THEN 1 ELSE 0 END) / COUNT(*), 2) as churn_rate_percent
FROM customers
GROUP BY Contract
ORDER BY churn_rate_percent DESC;
"""

result = pd.read_sql(query, conn)
print("Churn Rate by Contract Type:")
print(result.to_string(index=False))
conn.close()

Churn Rate by Contract Type:
      Contract  total_customers  churned_customers  churn_rate_percent
Month-to-month             3875               1655               42.71
      One year             1473                166               11.27
      Two year             1695                 48                2.83


In [7]:
# Cell 6: Check dataset size
print(f"Total rows: {len(df)}")
print(f"Total churned: {df[df['Churn'] == 'Yes'].shape[0]}")
print(f"Overall churn rate: {round(100 * df[df['Churn'] == 'Yes'].shape[0] / len(df), 2)}%")

Total rows: 7043
Total churned: 1869
Overall churn rate: 26.54%


In [5]:
# Cell 7: SQL Query - Churn rate by tenure bucket
conn = sqlite3.connect('../data/churn.db')

query = """
SELECT 
    CASE 
        WHEN tenure BETWEEN 0 AND 12 THEN '0-12 months'
        WHEN tenure BETWEEN 13 AND 24 THEN '13-24 months'
        WHEN tenure BETWEEN 25 AND 48 THEN '25-48 months'
        ELSE '48+ months'
    END as tenure_bucket,
    COUNT(*) as total_customers,
    SUM(CASE WHEN Churn = 'Yes' THEN 1 ELSE 0 END) as churned,
    ROUND(100.0 * SUM(CASE WHEN Churn = 'Yes' THEN 1 ELSE 0 END) / COUNT(*), 2) as churn_rate
FROM customers
GROUP BY tenure_bucket
ORDER BY 
    CASE tenure_bucket
        WHEN '0-12 months' THEN 1
        WHEN '13-24 months' THEN 2
        WHEN '25-48 months' THEN 3
        ELSE 4
    END;
"""

result = pd.read_sql(query, conn)
print("Churn Rate by Customer Tenure:")
print(result.to_string(index=False))
conn.close()

Churn Rate by Customer Tenure:
tenure_bucket  total_customers  churned  churn_rate
  0-12 months             2186     1037       47.44
 13-24 months             1024      294       28.71
 25-48 months             1594      325       20.39
   48+ months             2239      213        9.51


In [6]:
# Cell 8: SQL Query - Churn by Payment Method
conn = sqlite3.connect('../data/churn.db')

query = """
SELECT 
    PaymentMethod,
    COUNT(*) as total_customers,
    SUM(CASE WHEN Churn = 'Yes' THEN 1 ELSE 0 END) as churned,
    ROUND(100.0 * SUM(CASE WHEN Churn = 'Yes' THEN 1 ELSE 0 END) / COUNT(*), 2) as churn_rate
FROM customers
GROUP BY PaymentMethod
ORDER BY churn_rate DESC;
"""

result = pd.read_sql(query, conn)
print("Churn Rate by Payment Method:")
print(result.to_string(index=False))
conn.close()

Churn Rate by Payment Method:
            PaymentMethod  total_customers  churned  churn_rate
         Electronic check             2365     1071       45.29
             Mailed check             1612      308       19.11
Bank transfer (automatic)             1544      258       16.71
  Credit card (automatic)             1522      232       15.24


In [7]:
# Cell 9: SQL Query - Impact of additional services on churn
conn = sqlite3.connect('../data/churn.db')

query = """
SELECT 
    CASE 
        WHEN InternetService = 'No' THEN 'No internet'
        ELSE 'Has internet'
    END as internet_status,
    COUNT(*) as total_customers,
    SUM(CASE WHEN Churn = 'Yes' THEN 1 ELSE 0 END) as churned,
    ROUND(100.0 * SUM(CASE WHEN Churn = 'Yes' THEN 1 ELSE 0 END) / COUNT(*), 2) as churn_rate
FROM customers
GROUP BY internet_status;
"""

result = pd.read_sql(query, conn)
print("Churn Rate by Internet Service Status:")
print(result.to_string(index=False))
conn.close()

Churn Rate by Internet Service Status:
internet_status  total_customers  churned  churn_rate
   Has internet             5517     1756       31.83
    No internet             1526      113        7.40


In [8]:
# Cell 10: SQL Query - Monthly revenue at risk from churned customers
conn = sqlite3.connect('../data/churn.db')

query = """
SELECT 
    SUM(MonthlyCharges) as total_monthly_revenue,
    SUM(CASE WHEN Churn = 'Yes' THEN MonthlyCharges ELSE 0 END) as revenue_at_risk,
    ROUND(100.0 * SUM(CASE WHEN Churn = 'Yes' THEN MonthlyCharges ELSE 0 END) / SUM(MonthlyCharges), 2) as revenue_risk_percent
FROM customers;
"""

result = pd.read_sql(query, conn)
print("Monthly Revenue at Risk:")
print(result.to_string(index=False))
conn.close()

Monthly Revenue at Risk:
 total_monthly_revenue  revenue_at_risk  revenue_risk_percent
              456116.6        139130.85                  30.5


In [9]:
# Cell 11: Save all insights to a CSV file
import pandas as pd

# Create summary dataframe
summary_data = {
    'Metric': [
                'Total Customers',
                'Total Churned', 
                'Overall Churn Rate',
                'Month-to-month Churn Rate',
                'One-year Contract Churn Rate',
                'Two-year Contract Churn Rate',
                'First-year Customer Churn Rate',
                'Electronic Check Churn Rate',
                'Auto-pay Average Churn Rate',
                'Internet Users Churn Rate',
                'Monthly Revenue at Risk (₹)',
                'Revenue Risk Percentage'
    ],
    'Value': [
                '7,043',
                '1,869',
                '26.54%',
                '42.71%',
                '11.27%',
                '2.83%',
                '47.44%',
                '45.29%',
                '15.98%',
                '31.83%',
                '₹1,39,131',
                '30.5%'
    ]
}

summary_df = pd.DataFrame(summary_data)
summary_df.to_csv('../data/churn_insights_summary.csv', index=False)
print("Insights saved to: ../data/churn_insights_summary.csv")
print("\nYour Complete Summary:")
print(summary_df.to_string(index=False))

Insights saved to: ../data/churn_insights_summary.csv

Your Complete Summary:
                        Metric     Value
               Total Customers     7,043
                 Total Churned     1,869
            Overall Churn Rate    26.54%
     Month-to-month Churn Rate    42.71%
  One-year Contract Churn Rate    11.27%
  Two-year Contract Churn Rate     2.83%
First-year Customer Churn Rate    47.44%
   Electronic Check Churn Rate    45.29%
   Auto-pay Average Churn Rate    15.98%
     Internet Users Churn Rate    31.83%
   Monthly Revenue at Risk (₹) ₹1,39,131
       Revenue Risk Percentage     30.5%


In [10]:
# Cell 12: Export all query results for Power BI
import pandas as pd
import sqlite3

conn = sqlite3.connect('../data/churn.db')

# Query 1: Contract type churn
query1 = """
SELECT 
    Contract,
    COUNT(*) as total_customers,
    SUM(CASE WHEN Churn = 'Yes' THEN 1 ELSE 0 END) as churned_customers,
    ROUND(100.0 * SUM(CASE WHEN Churn = 'Yes' THEN 1 ELSE 0 END) / COUNT(*), 2) as churn_rate
FROM customers
GROUP BY Contract
ORDER BY churn_rate DESC;
"""
df1 = pd.read_sql(query1, conn)
df1.to_csv('../data/01_contract_churn.csv', index=False)

# Query 2: Tenure bucket churn
query2 = """
SELECT 
    CASE 
        WHEN tenure BETWEEN 0 AND 12 THEN '0-12 months'
        WHEN tenure BETWEEN 13 AND 24 THEN '13-24 months'
        WHEN tenure BETWEEN 25 AND 48 THEN '25-48 months'
        ELSE '48+ months'
    END as tenure_bucket,
    COUNT(*) as total_customers,
    SUM(CASE WHEN Churn = 'Yes' THEN 1 ELSE 0 END) as churned_customers,
    ROUND(100.0 * SUM(CASE WHEN Churn = 'Yes' THEN 1 ELSE 0 END) / COUNT(*), 2) as churn_rate
FROM customers
GROUP BY tenure_bucket;
"""
df2 = pd.read_sql(query2, conn)
df2.to_csv('../data/02_tenure_churn.csv', index=False)

# Query 3: Payment method churn
query3 = """
SELECT 
    PaymentMethod,
    COUNT(*) as total_customers,
    SUM(CASE WHEN Churn = 'Yes' THEN 1 ELSE 0 END) as churned_customers,
    ROUND(100.0 * SUM(CASE WHEN Churn = 'Yes' THEN 1 ELSE 0 END) / COUNT(*), 2) as churn_rate
FROM customers
GROUP BY PaymentMethod
ORDER BY churn_rate DESC;
"""
df3 = pd.read_sql(query3, conn)
df3.to_csv('../data/03_payment_churn.csv', index=False)

# Query 4: Internet service churn
query4 = """
SELECT 
    CASE 
        WHEN InternetService = 'No' THEN 'No Internet'
        ELSE 'Has Internet'
    END as internet_status,
    COUNT(*) as total_customers,
    SUM(CASE WHEN Churn = 'Yes' THEN 1 ELSE 0 END) as churned_customers,
    ROUND(100.0 * SUM(CASE WHEN Churn = 'Yes' THEN 1 ELSE 0 END) / COUNT(*), 2) as churn_rate
FROM customers
GROUP BY internet_status;
"""
df4 = pd.read_sql(query4, conn)
df4.to_csv('../data/04_internet_churn.csv', index=False)

# Raw data for detailed analysis
df_raw = pd.read_sql("SELECT * FROM customers", conn)
df_raw.to_csv('../data/00_raw_customer_data.csv', index=False)

conn.close()

print("All 5 CSV files exported to the 'data' folder:")
print("   - 00_raw_customer_data.csv (full dataset)")
print("   - 01_contract_churn.csv")
print("   - 02_tenure_churn.csv")
print("   - 03_payment_churn.csv")
print("   - 04_internet_churn.csv")

All 5 CSV files exported to the 'data' folder:
   - 00_raw_customer_data.csv (full dataset)
   - 01_contract_churn.csv
   - 02_tenure_churn.csv
   - 03_payment_churn.csv
   - 04_internet_churn.csv
